In [1]:
import torch

In [7]:
import numpy as np

num_samples_per_class = 1000
negative_samples = np.random.multivariate_normal(
    mean=[0, 3], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)
positive_samples = np.random.multivariate_normal(
    mean=[3, 0], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)
inputs = np.vstack((negative_samples, positive_samples)).astype(np.float32)

In [2]:
input_dim = 2
output_dim = 1

W = torch.rand(input_dim, output_dim, requires_grad=True)
b = torch.zeros(output_dim, requires_grad=True)

tensor([[0.4285],
        [0.2763]], requires_grad=True) tensor([0.], requires_grad=True)


In [3]:
def model(inputs, W, b):
    return torch.matmul(inputs, W) + b

In [4]:
def mean_squared_error(targets, prediction):
    losses = torch.square(targets - prediction)
    return torch.mean(losses)

In [5]:
learning_rate = 0.1

def training_step(inputs, targets, W, b):
    predictions = model(inputs, W, b)
    loss = mean_squared_error(targets, predictions)
    loss.backward()
    grad_w, grad_b = W.grad, b.grad
    with torch.no_grad():
        W -= grad_w * learning_rate
        b -= grad_b * learning_rate
    W.grad = None
    b.grad = None
    return loss

In [11]:
class LinearModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.W = torch.nn.Parameter(torch.rand(input_dim, output_dim))
        self.b = torch.nn.Parameter(torch.rand(output_dim))

    def forward(self, inputs):
        return torch.matmul(inputs, self.W) + self.b

In [13]:
model = LinearModel()

In [14]:
torch_inputs = torch.tensor(inputs)
output = model(torch_inputs)

In [15]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [16]:
def training_step_with_model(inputs, targets):
    predictions = model(inputs)
    loss = mean_squared_error(targets, predictions)
    loss.backward()
    optimizer.step()
    model.zero_grad()
    return loss

In [17]:
# Dynamo is the name of the compiler
compiled_model = torch.compile(model)

@torch.compile
def dense(inputs, W, b):
    return torch.relu(torch.matmul(inputs, W) + b)